In [2]:
# load necessary libraries

import os

os.environ.setdefault("SPCONV_ALGO", "native")
os.environ.setdefault("ATTN_BACKEND", "flash_attn")
os.environ.setdefault("TORCH_HOME", os.path.expanduser("~/.cache/torch"))

import torch
import numpy as np
from PIL import Image
from pytorch3d.ops import cubify
import trimesh

from dvd import DVDImageToVoxelPipeline, as_voxel_output


[SPARSE] Backend: spconv, Attention: flash_attn
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
# set device and load the BSP fine-tuned DVD image editing pipeline

device = torch.device("cuda")

# from local files

# dvd_pipeline = DVDImageToVoxelPipeline.from_files(
#     "./ckpts/dvd_img_BSP_ft.json",
#     "./ckpts/dvd_img_BSP_ft.safetensors",
#     device=device,
# )

# or from pretrained
dvd_pipeline = DVDImageToVoxelPipeline.from_pretrained("Zhengrui/dvd",variant="bsp", device="cuda")

Using cache found in /homes/zx1321/.cache/torch/hub/facebookresearch_dinov2_main


In [ ]:
# load a voxel grid in DVD coordinate convention and visualize it
coord = np.load("./assets/example_voxel_edit/voxel64_typical_building_mushroom_dis.npy")
voxels = as_voxel_output(torch.from_numpy(coord), resolution=64)
samples = voxels.samples.to(device)

cubified_meshes = cubify(samples.float(), 0.5, align="center")
mesh = trimesh.Trimesh(
    vertices=cubified_meshes.verts_packed().cpu().numpy(),
    faces=cubified_meshes.faces_packed().cpu().numpy(),
)
mesh.show()


## Edit with alternative condition

In [ ]:
# perturb the roof part of the generated shape
# The DVD edit sampler preserves voxels where keep_mask=True and regenerates where keep_mask=False.
edit_samples = samples.clone().long()
noise = torch.randint(0, 2, edit_samples.shape, device=device)
edit_samples[:, :, 28:, :] = noise[:, :, 28:, :]
keep_mask = torch.ones_like(edit_samples, dtype=torch.bool)
keep_mask[:, :, 28:, :] = False

# visualize perturbed mesh
cubified_meshes = cubify(edit_samples.float(), 0.5, align="center")
mesh = trimesh.Trimesh(
    vertices=cubified_meshes.verts_packed().cpu().numpy(),
    faces=cubified_meshes.faces_packed().cpu().numpy(),
)
mesh.show()


In [ ]:
# load the alternative image condition
image_path = "./assets/example_image_edit/flower_rm.png"
image = Image.open(image_path)
image


In [ ]:
# The DVD pipeline now obtains the image condition internally.
voxels_to_edit = as_voxel_output(edit_samples, resolution=64)
print(voxels_to_edit.samples.shape, keep_mask.shape)

res = dvd_pipeline.edit_voxels(
    image,
    voxels_to_edit,
    keep_mask=keep_mask,
    seed=0,
    steps=128,
    cfg_strength=0.45,
    preprocess_image=True,
    verbose=True,
)


In [ ]:
edited_samples = res.samples.to(device)
cubified_meshes = cubify(edited_samples.float(), 0.5, align="center")
mesh = trimesh.Trimesh(
    vertices=cubified_meshes.verts_packed().cpu().numpy(),
    faces=cubified_meshes.faces_packed().cpu().numpy(),
)
mesh.show()
